In [0]:
# -----------------------------------------
# Gold Layer - Customer Demographics Summary
# -----------------------------------------

from pyspark.sql import functions as F

# Source (Silver) and Target (Gold)
silver_table = "workspace.default.silver_customer"
gold_table = "workspace.default.gold_customer_demographics"

# Read Silver Data
df = spark.table(silver_table)

# Business Aggregations
gold_df = (
    df.groupBy(
        "continent",
        "country",
        "gender"
    )
    .agg(
        F.countDistinct("customerID").alias("customer_count")
    )
    .withColumn(
        "gold_processed_ts",
        F.current_timestamp()
    )
)

# Write Gold Table
(
    gold_df.write
           .format("delta")
           .mode("overwrite")
           .saveAsTable(gold_table)
)

# Validation
record_count = spark.table(gold_table).count()

print(f"Gold Load Completed Successfully")
print(f"Records Loaded: {record_count}")

display(gold_df)